# Ensemble Diversity vs. Efficiency Gain – Correlation Analysis

**Goal:** Determine which diversity metrics best predict the benefit of ensembling.

Steps:
1. Load ensemble results from disk.
2. Calculate **Efficiency Gain** = Ensemble Accuracy − Average Component Accuracy.
3. Compute Pearson correlation between each diversity metric and the efficiency gain.
4. Rank metrics by correlation strength.

## Imports

In [22]:
import torch
import os
import glob
import numpy as np
from scipy.stats import pearsonr

## Step 1 – Load Ensemble Results

Each `.pt` file stores one ensemble's accuracy, per-member accuracies, and diversity metrics.

In [23]:
save_dir = "../save/ensembles_test"

files = glob.glob(os.path.join(save_dir, "ensemble_*.pt"))
print(f"Found {len(files)} ensemble files in {save_dir}")

data_list = []
for f in files:
    try:
        entry = torch.load(f, map_location="cpu", weights_only=False)
        data_list.append(entry)
    except Exception as e:
        print(f"Error loading {f}: {e}")

print(f"Successfully loaded {len(data_list)} ensembles.")

Found 6 ensemble files in ../save/ensembles_test
Successfully loaded 6 ensembles.


## Step 2 – Compute Efficiency Gain

For each ensemble we compute:

$$\text{Efficiency Gain} = \text{Ensemble Accuracy} - \text{mean(Individual Accuracies)}$$

A positive gain means the ensemble outperforms its average member.

In [24]:
DIVERSITY_METRICS = [
    "div_pred_disagreement",
    "div_q_statistic",
    "div_double_fault",
    "div_jaccard",
    "div_cohens_kappa",
    "div_iou_top_n",
    "div_correctness_disagreement",
    "div_error_conditional_disagreement",
    "div_overall_agreement",
]

efficiency_gains = []
metric_values = {m: [] for m in DIVERSITY_METRICS}

for entry in data_list:
    # Skip entries missing required keys
    if not all(k in entry for k in ["acc", "individual_accuracies", "diversity"]):
        continue

    # Ensemble accuracy (prefer soft voting)
    acc_dict = entry["acc"]
    ens_acc = acc_dict.get("acc_soft", acc_dict.get("acc_hard"))
    if ens_acc is None:
        continue

    avg_indiv_acc = np.mean(entry["individual_accuracies"])
    gain = ens_acc - avg_indiv_acc

    # Collect diversity metrics (skip entry if any metric is missing or non-finite)
    div_dict = entry["diversity"]
    current = {}
    skip = False
    for m in DIVERSITY_METRICS:
        if m in div_dict and np.isfinite(div_dict[m]):
            current[m] = div_dict[m]
        else:
            skip = True
            break
    if skip:
        continue

    efficiency_gains.append(gain)
    for m in DIVERSITY_METRICS:
        metric_values[m].append(current[m])

print(f"Valid ensembles for analysis: {len(efficiency_gains)}")

Valid ensembles for analysis: 6


In [40]:
# Print a table of per-ensemble stats and diversities
import pandas as pd
from IPython.display import display

table_rows = []
for idx, entry in enumerate(data_list):
    # Skip entries missing required keys
    if not all(k in entry for k in ["acc", "individual_accuracies", "diversity"]):
        continue
    acc_dict = entry["acc"]
    ens_acc = acc_dict.get("acc_soft", acc_dict.get("acc_hard"))
    if ens_acc is None:
        continue
    indiv_accs = entry["individual_accuracies"]
    avg_indiv_acc = np.mean(indiv_accs)
    max_indiv_acc = np.max(indiv_accs)
    div_dict = entry["diversity"]
    # Only include if all diversity metrics are present and finite
    skip = False
    row = {
        "Ensemble #": idx + 1,
        "Avg Comp Acc": avg_indiv_acc,
        "Max Comp Acc": max_indiv_acc,
        "Ens Acc": ens_acc,
    }
    for m in DIVERSITY_METRICS:
        if m in div_dict and np.isfinite(div_dict[m]):
            row[m] = div_dict[m]
        else:
            skip = True
            break
    if skip:
        continue
    table_rows.append(row)
if table_rows:
    df = pd.DataFrame(table_rows)
    # Reorder columns for readability
    col_order = [
        "Ensemble #",
        "Avg Comp Acc",
        "Max Comp Acc",
        "Ens Acc",
    ] + DIVERSITY_METRICS
    df = df[col_order]
    display(df.style.format(precision=4))
else:
    print("No valid ensembles to display.")

# Print the models involved in each ensemble
for idx, entry in enumerate(data_list):
    print(f"Ensemble {idx+1}: {entry.get('run_names', 'N/A')[0]}")

# Sort with arbitrary order (e.g. by ensemble number) and print the table again
order = [6, 2, 1, 4, 3, 5]
if table_rows:
    df = pd.DataFrame(table_rows)
    df = df.iloc[[o - 1 for o in order]]  # Reorder rows based on specified order
    df = df[col_order]  # Ensure columns are in the same order
    display(df.style.format(precision=4))

,Ensemble #,Avg Comp Acc,Max Comp Acc,Ens Acc,div_pred_disagreement,div_q_statistic,div_double_fault,div_jaccard,div_cohens_kappa,div_iou_top_n,div_correctness_disagreement,div_error_conditional_disagreement,div_overall_agreement
0,1,0.9128,0.9160,0.9327,0.0906,0.9320,0.0485,0.9187,0.5140,0.6654,0.0773,0.6146,0.9227
1,2,0.9094,0.9163,0.9323,0.0966,0.9244,0.0494,0.9132,0.4994,0.7335,0.0825,0.6257,0.9175
2,3,0.9081,0.9104,0.9298,0.1007,0.9174,0.0491,0.9100,0.4874,0.6739,0.0856,0.6352,0.9144
3,4,0.9120,0.9153,0.9319,0.0953,0.9210,0.0469,0.9138,0.4886,0.6585,0.0821,0.6360,0.9179
4,5,0.9084,0.9147,0.9301,0.0984,0.9201,0.0492,0.9109,0.4910,0.6311,0.0847,0.6324,0.9153
5,6,0.9124,0.9165,0.9332,0.0925,0.9304,0.0485,0.9177,0.5098,0.6863,0.0783,0.6178,0.9217


Ensemble 1: crossentropy_wstopo_grid_256embdims_0.04rho_200epochs_512bsz_2nwork_0.002lr_0.5dropout___trial_00
Ensemble 2: crossentropy_wstopo_grid_256embdims_0.008rho_200epochs_512bsz_2nwork_0.002lr_0.5dropout___trial_00
Ensemble 3: crossentropy_wstopo_grid_256embdims_1.0rho_200epochs_512bsz_2nwork_0.002lr_0.5dropout___trial_00
Ensemble 4: crossentropy_wstopo_grid_256embdims_0.2rho_200epochs_512bsz_2nwork_0.002lr_0.5dropout___trial_00
Ensemble 5: crossentropy_wstopo_grid_256embdims_5.0rho_200epochs_512bsz_2nwork_0.002lr_0.5dropout___trial_00
Ensemble 6: crossentropy_wstopo_grid_256embdims_0.0rho_200epochs_512bsz_2nwork_0.002lr_0.5dropout___trial_00


,Ensemble #,Avg Comp Acc,Max Comp Acc,Ens Acc,div_pred_disagreement,div_q_statistic,div_double_fault,div_jaccard,div_cohens_kappa,div_iou_top_n,div_correctness_disagreement,div_error_conditional_disagreement,div_overall_agreement
5,6,0.9124,0.9165,0.9332,0.0925,0.9304,0.0485,0.9177,0.5098,0.6863,0.0783,0.6178,0.9217
1,2,0.9094,0.9163,0.9323,0.0966,0.9244,0.0494,0.9132,0.4994,0.7335,0.0825,0.6257,0.9175
0,1,0.9128,0.9160,0.9327,0.0906,0.9320,0.0485,0.9187,0.5140,0.6654,0.0773,0.6146,0.9227
3,4,0.9120,0.9153,0.9319,0.0953,0.9210,0.0469,0.9138,0.4886,0.6585,0.0821,0.6360,0.9179
2,3,0.9081,0.9104,0.9298,0.1007,0.9174,0.0491,0.9100,0.4874,0.6739,0.0856,0.6352,0.9144
4,5,0.9084,0.9147,0.9301,0.0984,0.9201,0.0492,0.9109,0.4910,0.6311,0.0847,0.6324,0.9153


## Step 3 – Pearson Correlation with Efficiency Gain

In [26]:
assert len(efficiency_gains) >= 2, "Need at least 2 data points for correlation."

results = []
for m in DIVERSITY_METRICS:
    corr, p_value = pearsonr(metric_values[m], efficiency_gains)
    results.append((m, corr, p_value))

## Step 4 – Rank Metrics by Predictive Power

In [27]:
results.sort(key=lambda x: x[1], reverse=True)

print(f"{'Metric':<40} | {'Correlation':>11} | {'P-Value':>11}")
print("-" * 70)
for m, r, p in results:
    print(f"{m:<40} | {r:>11.4f} | {p:>11.4e}")

Metric                                   | Correlation |     P-Value
----------------------------------------------------------------------
div_double_fault                         |      0.7969 |  5.7660e-02
div_pred_disagreement                    |      0.6321 |  1.7817e-01
div_correctness_disagreement             |      0.5704 |  2.3719e-01
div_iou_top_n                            |      0.5369 |  2.7201e-01
div_error_conditional_disagreement       |      0.2321 |  6.5817e-01
div_cohens_kappa                         |     -0.2882 |  5.7961e-01
div_q_statistic                          |     -0.3865 |  4.4912e-01
div_overall_agreement                    |     -0.5704 |  2.3719e-01
div_jaccard                              |     -0.5850 |  2.2265e-01
